In [ ]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv('transaksi_keuangan_sintetis_v1.csv')

# Melihat info umum dan jumlah data kosong (missing values)
print("Info Dataset:")
df.info()

# Melihat 5 data pertama
display(df.head())

Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18888 entries, 0 to 18887
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         18888 non-null  int64  
 1   date            18888 non-null  object 
 2   amount          18888 non-null  float64
 3   type            18888 non-null  object 
 4   category        17468 non-null  object 
 5   subcategory     17468 non-null  object 
 6   payment_method  18888 non-null  object 
 7   description     17464 non-null  object 
 8   balance_after   18888 non-null  float64
dtypes: float64(2), int64(1), object(6)
memory usage: 1.3+ MB


,user_id,date,amount,type,category,subcategory,payment_method,description,balance_after
0,2,2025-07-10,211095.59,expense,Food,go-food,cash,Pembayaran go-food,21120994.72
1,12,2025-04-07,138862.97,expense,ENTERTAINMENT,game topup,e-wallet,Pembayaran game topup,3239180.38
2,20,2025-07-03,8403741.89,income,Salary,gaji pokok,e-wallet,Penerimaan gaji pokok,-15951752.25
3,35,2025-01-26,485576.97,expense,Shopping,ELEKTRONIK,cash,Pembayaran elektronik,1268865.03
4,22,2025-04-22,198081.00,expense,Food,restoran,cash,NaN,-1739966.76


In [ ]:
# Cek duplikat
jumlah_duplikat = df.duplicated().sum()
print(f"Jumlah baris duplikat ditemukan: {jumlah_duplikat}")

# Menghapus duplikat dan mereset index
df = df.drop_duplicates().reset_index(drop=True)
print(f"Jumlah baris setelah hapus duplikat: {len(df)}")

Jumlah baris duplikat ditemukan: 433
Jumlah baris setelah hapus duplikat: 18455


In [ ]:
kolom_teks = ['category', 'subcategory', 'payment_method']

for col in kolom_teks:
    # Ubah ke string, lalu ubah jadi title case (ex: 'FOOD' -> 'Food', 'ojek ONLINE' -> 'Ojek Online')
    df[col] = df[col].astype(str).str.title()
    # Kembalikan string 'Nan' menjadi tipe data np.nan sesungguhnya agar terbaca sebagai missing value
    df[col] = df[col].replace('Nan', np.nan)

In [ ]:
# Cek banyak 'category' dan 'subcategory' dalam dataset

print("Unique values in 'category':")
print(df['category'].unique())
print(f"Number of unique categories: {df['category'].nunique()}\n")

print("Unique values in 'subcategory':")
print(df['subcategory'].unique())
print(f"Number of unique subcategories: {df['subcategory'].nunique()}")

Unique values in 'category':
['Food' 'Entertainment' 'Salary' 'Shopping' 'Bills' nan 'Transport'
 'Others']
Number of unique categories: 7

Unique values in 'subcategory':
['Go-Food' 'Game Topup' 'Gaji Pokok' 'Elektronik' 'Restoran' 'Bioskop' nan
 'Air' 'Minimarket' 'Servis' 'Warung' 'Ojek Online' 'Tol' 'Sedekah'
 'Pulsa' 'Hadiah' 'Listrik' 'Pakaian' 'Bensin' 'Konser' 'Parkir' 'Bonus'
 'Kopi' 'Peralatan Rumah' 'Netflix' 'Insentif' 'Skincare' 'Hobi'
 'Asuransi' 'Internet' 'Lain-Lain' 'Biaya Admin' 'Spotify' 'Freelance']
Number of unique subcategories: 33


In [ ]:
# Cek Missing Values

print("Missing Values sebelum cleaning:")
print(df.isnull().sum())

# 1. Isi missing value di 'description' dengan teks default
df['description'] = df['description'].fillna('Tidak ada deskripsi')

# 2. Isi 'category' dan 'subcategory' yang kosong dengan 'Unknown'
df['category'] = df['category'].fillna('Others')
df['subcategory'] = df['subcategory'].fillna('Lain-Lain')

print("\nMissing Values setelah cleaning:")
print(df.isnull().sum())

Missing Values sebelum cleaning:
user_id              0
date                 0
amount               0
type                 0
category          1376
subcategory       1384
payment_method       0
description       1381
balance_after        0
dtype: int64

Missing Values setelah cleaning:
user_id           0
date              0
amount            0
type              0
category          0
subcategory       0
payment_method    0
description       0
balance_after     0
dtype: int64


In [ ]:
# Membuang outliers

def cap_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Batas bawah tidak boleh kurang dari 0
    # Asumsi minimum transaksi wajar adalah Rp 1000
    lower_bound = max(1000, lower_bound)

    # Terapkan capping (nilai di luar batas akan ditarik ke batas tersebut)
    return np.clip(series, lower_bound, upper_bound)

# Terapkan capping untuk expense dan income secara terpisah
df.loc[df['type'] == 'expense', 'amount'] = cap_outliers_iqr(df.loc[df['type'] == 'expense', 'amount'])
df.loc[df['type'] == 'income', 'amount'] = cap_outliers_iqr(df.loc[df['type'] == 'income', 'amount'])

print("Outliers berhasil ditangani dengan metode Capping IQR.")

Outliers berhasil ditangani dengan metode Capping IQR.


In [ ]:
# Cek data dengan amount dibawah 2000
data_under_2000 = df[df['amount'] < 2000]
print("Data dengan amount dibawah 2000:")
data_under_2000.head()

Data dengan amount dibawah 2000:


,user_id,date,amount,type,category,subcategory,payment_method,description,balance_after
133,22,2025-01-03,1000.0,expense,Entertainment,Game Topup,Cash,Pembayaran game topup,2489419.85
368,41,2025-09-10,1000.0,expense,Food,Restoran,Debit,Tidak ada deskripsi,3062649.78
490,27,2025-07-28,1000.0,expense,Food,Kopi,Debit,Pembayaran kopi,-30616259.45
501,27,2025-07-31,1000.0,expense,Entertainment,Konser,Cash,Pembayaran konser,-31589692.38
844,29,2025-02-05,1000.0,expense,Transport,Parkir,E-Wallet,Pembayaran parkir,13234536.57


In [ ]:
# hapus data dengan amount dibawah 2000
df = df[df['amount'] >= 2000]
df.reset_index(drop=True, inplace=True)
print("Data berhasil dihapus.")

Data berhasil dihapus.


In [ ]:
# drop fitur balance_after (tidak terpakai)
df = df.drop(columns=['balance_after'])

In [ ]:
### Membulatkan Nilai Transaksi (Amount)

# Membulatkan kolom amount ke ratusan terdekat
# Caranya: bagi 100, bulatkan, kalikan 100 lagi
df['amount'] = (df['amount'] / 100).round() * 100

print("Beberapa nilai 'amount' setelah dibulatkan ke ratusan terdekat:")
display(df[['type', 'category', 'amount']].head())

Beberapa nilai 'amount' setelah dibulatkan ke ratusan terdekat:


,type,category,amount
0,expense,Food,211100.0
1,expense,Entertainment,138900.0
2,income,Salary,8403700.0
3,expense,Shopping,485600.0
4,expense,Food,198100.0


In [ ]:
# Memperbaiki beberapa harga (amount)

# 1. Memperbaiki harga Spotify
# Mengubah semua transaksi Spotify menjadi harga Premium (misal: Rp 27.500 untuk pelajar)
df.loc[df['subcategory'] == 'Spotify', 'amount'] = 27500

# 2. Memperbaiki Biaya Admin
# Mengubah biaya admin menjadi tarif tetap (misal: Rp 6.500 untuk admin antar bank)
df.loc[df['subcategory'] == 'Biaya Admin', 'amount'] = 6500

# 3. Memperbaiki harga Netflix
# Mengubah biaya admin menjadi tarif tetap (misal: Rp 6.500 untuk admin antar bank)
df.loc[df['subcategory'] == 'Netflix', 'amount'] = 120000

# 4. Memperbaiki Biaya Parkir
# Mencari semua baris dengan subcategory 'Parkir'
mask_parkir = df['subcategory'] == 'Parkir'
jumlah_parkir = mask_parkir.sum()

# Membuat ulang harga parkir dengan nominal yang masuk akal
tarif_parkir_masuk_akal = [2000, 3000, 4000, 5000, 10000]
df.loc[mask_parkir, 'amount'] = np.random.choice(tarif_parkir_masuk_akal, size=jumlah_parkir)

In [ ]:
file_bersih = 'transaksi_keuangan_clean.csv'
df.to_csv(file_bersih, index=False)
print(f"Data yang sudah dibersihkan berhasil disimpan ke: {file_bersih}")

Data yang sudah dibersihkan berhasil disimpan ke: transaksi_keuangan_clean.csv
